imports

In [1]:
import pandas as pd 
import numpy as np 
from pathlib import Path

loading 5 datasets

In [2]:
# Project root
PROJECT_ROOT = Path.cwd().parents[1]

# Raw data directory
DATA_DIR = PROJECT_ROOT / "data" / "raw"

# Load datasets
current_voltage = pd.read_csv(DATA_DIR / "CurrentVoltage.csv")
overview = pd.read_csv(DATA_DIR / "Overview.csv")
power = pd.read_csv(DATA_DIR / "Power.csv")
power_factor = pd.read_csv(DATA_DIR / "PowerFactor.csv")
total_power = pd.read_csv(DATA_DIR / "TotalPower.csv")

In [3]:
datasets = {
    "CurrentVoltage.csv": current_voltage,
    "Overview.csv": overview,
    "Power.csv": power,
    "PowerFactor.csv": power_factor,
    "TotalPower.csv": total_power,
}


def data_quality_report(filename, df):
    print("=" * 80)
    print(f"FILENAME: {filename}")
    print("=" * 80)

    # 1. Row count
    print(f"\nROW COUNT: {len(df)}")

    # 2. Column names
    print("\nCOLUMN NAMES:")
    for col in df.columns:
        print(f"  {col}")

    # 3. Dtypes
    print("\nDTYPES:")
    print(df.dtypes.to_string())

    # 4. Timestamp column
    timestamp_column = "DeviceTimeStamp"

    if timestamp_column in df.columns:
        print(f"\nTIMESTAMP COLUMN: {timestamp_column}")

        # Convert only for analysis
        timestamps = pd.to_datetime(
            df[timestamp_column],
            errors="coerce"
        )

        # 5. Timestamp min/max
        print(f"TIMESTAMP MIN: {timestamps.min()}")
        print(f"TIMESTAMP MAX: {timestamps.max()}")

        # Invalid timestamps
        invalid_count = timestamps.isna().sum()
        print(f"INVALID TIMESTAMPS: {invalid_count}")

        # 6. Duplicate timestamps
        duplicate_count = timestamps.duplicated().sum()
        print(f"DUPLICATE TIMESTAMP ROWS: {duplicate_count}")

        duplicate_timestamps = timestamps[
            timestamps.duplicated(keep=False)
        ]

        print(
            f"DISTINCT DUPLICATE TIMESTAMPS: "
            f"{duplicate_timestamps.nunique()}"
        )

    else:
        print("\nTIMESTAMP COLUMN: NOT FOUND")

    # 7. Missing values
    print("\nMISSING VALUES:")

    missing = df.isna().sum()

    if missing.sum() == 0:
        print("  No missing values")
    else:
        for column, count in missing.items():
            if count > 0:
                print(f"  {column}: {count}")

    print()

In [4]:
for filename, df in datasets.items():
    data_quality_report(filename, df)

FILENAME: CurrentVoltage.csv

ROW COUNT: 19352

COLUMN NAMES:
  DeviceTimeStamp
  VL1
  VL2
  VL3
  IL1
  IL2
  IL3
  VL12
  VL23
  VL31
  INUT

DTYPES:
DeviceTimeStamp     object
VL1                float64
VL2                float64
VL3                float64
IL1                float64
IL2                float64
IL3                float64
VL12               float64
VL23               float64
VL31               float64
INUT               float64

TIMESTAMP COLUMN: DeviceTimeStamp
TIMESTAMP MIN: 2019-06-25 13:06:00
TIMESTAMP MAX: 2020-04-14 00:30:00
INVALID TIMESTAMPS: 0
DUPLICATE TIMESTAMP ROWS: 437
DISTINCT DUPLICATE TIMESTAMPS: 437

MISSING VALUES:
  No missing values

FILENAME: Overview.csv

ROW COUNT: 20316

COLUMN NAMES:
  DeviceTimeStamp
  OTI
  WTI
  ATI
  OLI
  OTI_A
  OTI_T
  MOG_A

DTYPES:
DeviceTimeStamp     object
OTI                float64
WTI                float64
ATI                float64
OLI                float64
OTI_A              float64
OTI_T              float64


investigating duplicates timestamp columns

In [5]:
def investigate_duplicates(filename, df):
    df = df.copy()

    # Convert timestamp
    df["DeviceTimeStamp"] = pd.to_datetime(
        df["DeviceTimeStamp"],
        errors="coerce"
    )

    # Find timestamps occurring more than once
    timestamp_counts = df["DeviceTimeStamp"].value_counts()

    duplicate_timestamp_values = timestamp_counts[
        timestamp_counts > 1
    ]

    print("=" * 80)
    print(f"{filename}")
    print("=" * 80)

    print(f"Distinct duplicated timestamps: {len(duplicate_timestamp_values)}")
    print(f"Rows belonging to duplicated timestamps: "
          f"{duplicate_timestamp_values.sum()}")
    print(f"Maximum occurrences of one timestamp: "
          f"{duplicate_timestamp_values.max()}")

    # Check whether duplicate timestamp groups have identical measurements
    non_timestamp_cols = [
        col for col in df.columns
        if col != "DeviceTimeStamp"
    ]

    identical_groups = 0
    conflicting_groups = 0

    for timestamp in duplicate_timestamp_values.index:

        group = df[df["DeviceTimeStamp"] == timestamp]

        # Check whether all measurement rows are identical
        if group[non_timestamp_cols].nunique(dropna=False).max() == 1:
            identical_groups += 1
        else:
            conflicting_groups += 1

    print(f"Duplicate groups with identical values: {identical_groups}")
    print(f"Duplicate groups with conflicting values: {conflicting_groups}")
    print()

In [6]:
for filename, df in datasets.items():
    investigate_duplicates(filename, df)

CurrentVoltage.csv
Distinct duplicated timestamps: 437
Rows belonging to duplicated timestamps: 874
Maximum occurrences of one timestamp: 2
Duplicate groups with identical values: 32
Duplicate groups with conflicting values: 405

Overview.csv
Distinct duplicated timestamps: 931
Rows belonging to duplicated timestamps: 1871
Maximum occurrences of one timestamp: 4
Duplicate groups with identical values: 568
Duplicate groups with conflicting values: 363

Power.csv
Distinct duplicated timestamps: 438
Rows belonging to duplicated timestamps: 876
Maximum occurrences of one timestamp: 2
Duplicate groups with identical values: 429
Duplicate groups with conflicting values: 9

PowerFactor.csv
Distinct duplicated timestamps: 431
Rows belonging to duplicated timestamps: 862
Maximum occurrences of one timestamp: 2
Duplicate groups with identical values: 1
Duplicate groups with conflicting values: 430

TotalPower.csv
Distinct duplicated timestamps: 406
Rows belonging to duplicated timestamps: 812
Ma

fixing duplicates

In [11]:
def duplicate_details(filename, df, max_groups=10):
    df = df.copy()

    # Convert timestamp
    df["DeviceTimeStamp"] = pd.to_datetime(
        df["DeviceTimeStamp"],
        errors="coerce"
    )

    # Find duplicated timestamps
    counts = df["DeviceTimeStamp"].value_counts()
    duplicate_timestamps = counts[counts > 1].index

    print("=" * 90)
    print(f"{filename}")
    print("=" * 90)

    shown = 0

    for timestamp in duplicate_timestamps:

        group = df[df["DeviceTimeStamp"] == timestamp]

        print(f"\nTimestamp: {timestamp}")
        print(f"Occurrences: {len(group)}")

        # Check which columns actually differ
        differing_columns = []

        for column in df.columns:
            if column == "DeviceTimeStamp":
                continue

            if group[column].nunique(dropna=False) > 1:
                differing_columns.append(column)

        if differing_columns:
            print("Different columns:")
            print(differing_columns)
        else:
            print("All measurements are identical.")

        print("\nRows:")
        print(group.to_string(index=False))

        shown += 1

        if shown >= max_groups:
            break

    print()

In [12]:
for filename, df in datasets.items():
    duplicate_details(filename, df, max_groups=10)

CurrentVoltage.csv

Timestamp: 2019-06-29 13:04:00
Occurrences: 2
Different columns:
['VL1', 'VL2', 'VL3']

Rows:
    DeviceTimeStamp   VL1   VL2   VL3  IL1  IL2  IL3  VL12  VL23  VL31  INUT
2019-06-29 13:04:00 244.1 244.1 244.2  0.0  0.0  0.0   0.0   0.0   0.0   0.0
2019-06-29 13:04:00 243.6 243.7 243.7  0.0  0.0  0.0   0.0   0.0   0.0   0.0

Timestamp: 2019-06-28 15:08:00
Occurrences: 2
Different columns:
['VL1', 'VL2', 'VL3']

Rows:
    DeviceTimeStamp   VL1   VL2   VL3  IL1  IL2  IL3  VL12  VL23  VL31  INUT
2019-06-28 15:08:00 237.5 237.5 237.6  0.0  0.0  0.0   0.0   0.0   0.0   0.0
2019-06-28 15:08:00 238.1 238.2 238.2  0.0  0.0  0.0   0.0   0.0   0.0   0.0

Timestamp: 2019-06-29 16:40:00
Occurrences: 2
Different columns:
['VL2']

Rows:
    DeviceTimeStamp   VL1   VL2   VL3  IL1  IL2  IL3  VL12  VL23  VL31  INUT
2019-06-29 16:40:00 245.1 245.1 245.2  0.0  0.0  0.0   0.0   0.0   0.0   0.0
2019-06-29 16:40:00 245.1 245.2 245.2  0.0  0.0  0.0   0.0   0.0   0.0   0.0

Timestamp: 2019-

In [9]:
def timestamp_interval_report(filename, df):
    temp = df.copy()

    temp["DeviceTimeStamp"] = pd.to_datetime(
        temp["DeviceTimeStamp"],
        errors="coerce"
    )

    temp = temp.sort_values("DeviceTimeStamp")

    intervals = temp["DeviceTimeStamp"].diff().dropna()

    print("=" * 80)
    print(filename)
    print("=" * 80)

    print("\nMost common timestamp intervals:")
    print(intervals.value_counts().head(10))

    print("\nTimestamp precision:")
    print(
        "All timestamps have zero seconds:",
        (temp["DeviceTimeStamp"].dt.second == 0).all()
    )

    print()

In [10]:
for filename, df in datasets.items():
    timestamp_interval_report(filename, df)

CurrentVoltage.csv

Most common timestamp intervals:
DeviceTimeStamp
0 days 00:15:00    16700
0 days 00:01:00      931
0 days 00:00:00      437
0 days 00:14:00      120
0 days 00:16:00       99
0 days 00:13:00       95
0 days 00:12:00       73
0 days 00:17:00       72
0 days 00:03:00       67
0 days 00:05:00       63
Name: count, dtype: int64

Timestamp precision:
All timestamps have zero seconds: True

Overview.csv

Most common timestamp intervals:
DeviceTimeStamp
0 days 00:15:00    16632
0 days 00:01:00     1096
0 days 00:00:00      940
0 days 00:03:00      132
0 days 00:05:00      132
0 days 00:14:00      131
0 days 00:02:00      111
0 days 00:07:00      105
0 days 00:04:00      104
0 days 00:08:00       98
Name: count, dtype: int64

Timestamp precision:
All timestamps have zero seconds: True

Power.csv

Most common timestamp intervals:
DeviceTimeStamp
0 days 00:15:00    16636
0 days 00:01:00      914
0 days 00:00:00      438
0 days 00:14:00      125
0 days 00:16:00      115
0 days 